In [1]:
import pandas as pd, numpy as np

panel = pd.read_parquet("../data/processed/weekly_demand.parquet")
panel["week"] = pd.to_datetime(panel["week"])
groups = pd.read_parquet("../data/processed/product_groups.parquet").set_index("StockCode")["group"]

wide = panel.pivot(index="week", columns="StockCode", values="units_regular").sort_index()
test_weeks = wide.index[-12:]
print(wide.shape, test_weeks[0].date(), test_weeks[-1].date())

(52, 60) 2011-09-12 2011-11-28


In [2]:
def forecast(kind, h):
    if kind == "naive":
        return wide.shift(h)
    if kind == "ma4":
        return wide.rolling(4).mean().shift(h)

def score(pred, cols=None):
    a = wide.loc[test_weeks]
    p = pred.loc[test_weeks]
    if cols is not None:
        a, p = a[cols], p[cols]
    err = (a - p).abs()
    return {"MAE": round(err.stack().mean(), 1),
            "WAPE": round(err.sum().sum() / a.sum().sum(), 3)}

rows = []
for h in [1, 2]:
    for kind in ["naive", "ma4"]:
        pred = forecast(kind, h)
        rows.append({"model": kind, "horizon": h, "group": "all", **score(pred)})
        for g in ["stable", "intermittent", "seasonal_ramp"]:
            cols = groups[groups == g].index
            rows.append({"model": kind, "horizon": h, "group": g, **score(pred, cols)})

results = pd.DataFrame(rows)
print(results.pivot_table(index=["group", "horizon"], columns="model", values="WAPE"))

model                    ma4  naive
group         horizon              
all           1        0.361  0.431
              2        0.380  0.442
intermittent  1        0.474  0.611
              2        0.497  0.618
seasonal_ramp 1        0.308  0.272
              2        0.370  0.340
stable        1        0.365  0.459
              2        0.370  0.453


In [3]:
results.to_csv("../data/processed/baseline_results.csv", index=False)

## Validation design
- Test period: last 12 weeks (12 Sep to 28 Nov 2011); training history before that.
- Rolling-origin: a forecast for week t uses only data up to week t-h (h = 1 and 2 weeks, standing in for supplier lead time).
- Metrics: MAE and WAPE (MAPE avoided because of zero-demand weeks).
- Baselines: naive and 4-week moving average. No seasonal-naive baseline because only one year of data exists.
- Results are reported for all products and for three groups: stable (45), seasonal ramp (9), intermittent (6).